In [1]:
import pandas as pd

#read in the html table
#since the actual headers are on the second row, using multi index
salary_table = pd.read_html('https://www.basketball-reference.com/contracts/players.html', header = [0, 1])

#select the table
contracts = salary_table[0]

In [2]:
#flatten multi index cols
contracts.columns = contracts.columns.get_level_values(1)

#identify col headers
print(contracts.columns)

Index(['Rk', 'Player', 'Tm', '2025-26', '2026-27', '2027-28', '2028-29',
       '2029-30', '2030-31', 'Guaranteed'],
      dtype='object')


In [3]:
#discovered that the table has "break" rows that don't match others
#dropping duplicate header rows
contracts = contracts[contracts['Player'] != 'Player']
contracts = contracts[contracts['Rk'] != 'Rk']

# Transformation 1: Standardize Player Names

## Create a new column that shows player name cleaned up to all lower letters and remove any punctuation

In [4]:
contracts['Player_clean'] = contracts['Player'].str.lower().str.strip().replace(',', '').str.replace('.', '')
contracts.head()

,Rk,Player,Tm,2025-26,2026-27,2027-28,2028-29,2029-30,2030-31,Guaranteed,Player_clean
0,1,Stephen Curry,GSW,"$59,606,817","$62,587,158",NaN,NaN,NaN,NaN,"$122,193,975",stephen curry
1,2,Joel Embiid,PHI,"$55,224,526","$57,985,752","$62,624,612","$67,263,472",NaN,NaN,"$175,834,890",joel embiid
2,3,Nikola Jokić,DEN,"$55,224,526","$59,033,114","$62,841,702",NaN,NaN,NaN,"$114,257,640",nikola jokić
3,4,Kevin Durant,HOU,"$54,708,609",NaN,NaN,NaN,NaN,NaN,"$54,708,609",kevin durant
4,5,Jayson Tatum,BOS,"$54,126,450","$58,456,566","$62,786,682","$67,116,798","$71,446,914",NaN,"$242,486,496",jayson tatum


# Transformation 2: Standardize contract salary amounts

## Remove any special characters ($, -, and commas) and replace NaNs with zeroes

In [5]:
salary_cols = ['2025-26', '2026-27', '2027-28', '2028-29', '2029-30', '2030-31', 'Guaranteed']

for col in salary_cols:
    contracts[col] = (
        contracts[col]
        .astype(str)
        .replace('[\$,]', '', regex=True)    
        .replace(['—', 'nan', 'Salary'], '0') 
        .astype(float)
    )

contracts.head()

<>:7: SyntaxWarning: invalid escape sequence '\$'
<>:7: SyntaxWarning: invalid escape sequence '\$'
/var/folders/dj/tlflpdsx4cjcpzn7v8wdcddc0000gn/T/ipykernel_95261/3664903393.py:7: SyntaxWarning: invalid escape sequence '\$'
  .replace('[\$,]', '', regex=True)


,Rk,Player,Tm,2025-26,2026-27,2027-28,2028-29,2029-30,2030-31,Guaranteed,Player_clean
0,1,Stephen Curry,GSW,59606817.0,62587158.0,0.0,0.0,0.0,0.0,122193975.0,stephen curry
1,2,Joel Embiid,PHI,55224526.0,57985752.0,62624612.0,67263472.0,0.0,0.0,175834890.0,joel embiid
2,3,Nikola Jokić,DEN,55224526.0,59033114.0,62841702.0,0.0,0.0,0.0,114257640.0,nikola jokić
3,4,Kevin Durant,HOU,54708609.0,0.0,0.0,0.0,0.0,0.0,54708609.0,kevin durant
4,5,Jayson Tatum,BOS,54126450.0,58456566.0,62786682.0,67116798.0,71446914.0,0.0,242486496.0,jayson tatum


In [6]:
#validate no special characters anymore in one of the cols
contracts['2025-26'].unique()

array([59606817., 55224526., 54708609., 54126450., 53666270., 53142264.,
       52627153., 51666090., 50000000., 47499660., 46394100.,        0.,
       45999660., 45550512., 43636000., 39568966., 39446090., 38661750.,
       38333050., 38095238., 37958760., 37096620., 36725670., 35550814.,
       35000000., 34944001., 33944954., 33333333., 33296296., 32500000.,
       32400000., 32000001., 31848215., 31557103., 31000000., 30935520.,
       30731707., 30666666., 30000000., 28500000., 28223215., 28100000.,
       27705357., 27678571., 27000000., 26643031., 26634146., 26580000.,
       25892857., 25352272., 25000000., 24900000., 24858622., 24570000.,
       23413395., 23303571., 22841455., 22410605., 21621500., 21586855.,
       21481481., 21124110., 20543478., 20482758., 20000000., 19500000.,
       19472240., 19375000., 19175000., 19000000., 18797619., 18259259.,
       18102000., 18080496., 18000000., 17991071., 17722222., 17500000.,
       16875000., 16666667., 16020000., 15936452., 

# Transformation 3: Sum yearly salaries

## Summing up the total contract value by adding all year columns (2025-26 through 2030-31)

In [7]:
contracts['Total Contract Value'] = contracts[salary_cols].sum(axis=1)
contracts.head()

,Rk,Player,Tm,2025-26,2026-27,2027-28,2028-29,2029-30,2030-31,Guaranteed,Player_clean,Total Contract Value
0,1,Stephen Curry,GSW,59606817.0,62587158.0,0.0,0.0,0.0,0.0,122193975.0,stephen curry,244387950.0
1,2,Joel Embiid,PHI,55224526.0,57985752.0,62624612.0,67263472.0,0.0,0.0,175834890.0,joel embiid,418933252.0
2,3,Nikola Jokić,DEN,55224526.0,59033114.0,62841702.0,0.0,0.0,0.0,114257640.0,nikola jokić,291356982.0
3,4,Kevin Durant,HOU,54708609.0,0.0,0.0,0.0,0.0,0.0,54708609.0,kevin durant,109417218.0
4,5,Jayson Tatum,BOS,54126450.0,58456566.0,62786682.0,67116798.0,71446914.0,0.0,242486496.0,jayson tatum,556419906.0


# Transformation 4: Compare Guaranteed against Total

## Determine the % of the contract that is guaranteed

In [8]:
contracts['Guaranteed %'] = (contracts['Guaranteed'] / contracts['Total Contract Value']) * 100
contracts.head()

,Rk,Player,Tm,2025-26,2026-27,2027-28,2028-29,2029-30,2030-31,Guaranteed,Player_clean,Total Contract Value,Guaranteed %
0,1,Stephen Curry,GSW,59606817.0,62587158.0,0.0,0.0,0.0,0.0,122193975.0,stephen curry,244387950.0,50.000000
1,2,Joel Embiid,PHI,55224526.0,57985752.0,62624612.0,67263472.0,0.0,0.0,175834890.0,joel embiid,418933252.0,41.972054
2,3,Nikola Jokić,DEN,55224526.0,59033114.0,62841702.0,0.0,0.0,0.0,114257640.0,nikola jokić,291356982.0,39.215686
3,4,Kevin Durant,HOU,54708609.0,0.0,0.0,0.0,0.0,0.0,54708609.0,kevin durant,109417218.0,50.000000
4,5,Jayson Tatum,BOS,54126450.0,58456566.0,62786682.0,67116798.0,71446914.0,0.0,242486496.0,jayson tatum,556419906.0,43.579767


# Transformation 5: Create a new column for whole team name from abbreviation

In [9]:
#Step 1: get the abbreviated team names used in the dataset
unique_values = contracts['Tm'].unique()
unique_values

array(['GSW', 'PHI', 'DEN', 'HOU', 'BOS', 'MIL', 'DAL', 'PHO', 'NYK',
       'LAL', 'LAC', 'SAC', 'CLE', 'UTA', nan, 'ATL', 'MIN', 'IND', 'NOP',
       'MEM', 'TOR', 'ORL', 'DET', 'OKC', 'BRK', 'CHO', 'MIA', 'SAS',
       'POR', 'WAS', 'CHI'], dtype=object)

In [10]:
#Step 2: created a dictionary for the full team names, ignoring nan
team_full_name = {
    'ATL': 'Atlanta Hawks',
    'BOS': 'Boston Celtics',
    'BRK': 'Brooklyn Nets',
    'CHO': 'Charlotte Hornets',
    'CHI': 'Chicago Bulls',
    'CLE': 'Cleveland Cavaliers',
    'DAL': 'Dallas Mavericks',
    'DEN': 'Denver Nuggets',
    'DET': 'Detroit Pistons',
    'GSW': 'Golden State Warriors',
    'HOU': 'Houston Rockets',
    'IND': 'Indiana Pacers',
    'LAC': 'Los Angeles Clippers',
    'LAL': 'Los Angeles Lakers',
    'MEM': 'Memphis Grizzlies',
    'MIA': 'Miami Heat',
    'MIL': 'Milwaukee Bucks',
    'MIN': 'Minnesota Timberwolves',
    'NOP': 'New Orleans Pelicans',
    'NYK': 'New York Knicks',
    'OKC': 'Oklahoma City Thunder',
    'ORL': 'Orlando Magic',
    'PHI': 'Philadelphia 76ers',
    'PHO': 'Phoenix Suns',
    'POR': 'Portland Trail Blazers',
    'SAC': 'Sacramento Kings',
    'SAS': 'San Antonio Spurs',
    'TOR': 'Toronto Raptors',
    'UTA': 'Utah Jazz',
    'WAS': 'Washington Wizards'
}

#Step 3: create new column with team full names from dictionary
contracts['Whole Team Name'] = contracts['Tm'].map(team_full_name)
contracts.head()

,Rk,Player,Tm,2025-26,2026-27,2027-28,2028-29,2029-30,2030-31,Guaranteed,Player_clean,Total Contract Value,Guaranteed %,Whole Team Name
0,1,Stephen Curry,GSW,59606817.0,62587158.0,0.0,0.0,0.0,0.0,122193975.0,stephen curry,244387950.0,50.000000,Golden State Warriors
1,2,Joel Embiid,PHI,55224526.0,57985752.0,62624612.0,67263472.0,0.0,0.0,175834890.0,joel embiid,418933252.0,41.972054,Philadelphia 76ers
2,3,Nikola Jokić,DEN,55224526.0,59033114.0,62841702.0,0.0,0.0,0.0,114257640.0,nikola jokić,291356982.0,39.215686,Denver Nuggets
3,4,Kevin Durant,HOU,54708609.0,0.0,0.0,0.0,0.0,0.0,54708609.0,kevin durant,109417218.0,50.000000,Houston Rockets
4,5,Jayson Tatum,BOS,54126450.0,58456566.0,62786682.0,67116798.0,71446914.0,0.0,242486496.0,jayson tatum,556419906.0,43.579767,Boston Celtics


In [11]:
#validate the changes in the new col
unique_values = contracts['Whole Team Name'].unique()
unique_values

array(['Golden State Warriors', 'Philadelphia 76ers', 'Denver Nuggets',
       'Houston Rockets', 'Boston Celtics', 'Milwaukee Bucks',
       'Dallas Mavericks', 'Phoenix Suns', 'New York Knicks',
       'Los Angeles Lakers', 'Los Angeles Clippers', 'Sacramento Kings',
       'Cleveland Cavaliers', 'Utah Jazz', nan, 'Atlanta Hawks',
       'Minnesota Timberwolves', 'Indiana Pacers', 'New Orleans Pelicans',
       'Memphis Grizzlies', 'Toronto Raptors', 'Orlando Magic',
       'Detroit Pistons', 'Oklahoma City Thunder', 'Brooklyn Nets',
       'Charlotte Hornets', 'Miami Heat', 'San Antonio Spurs',
       'Portland Trail Blazers', 'Washington Wizards', 'Chicago Bulls'],
      dtype=object)

# Transformation 6: Find and remove any remaining duplicate rows

In [12]:
#Step 1: determine the amount of duplicate rows
duplicates = contracts[contracts.duplicated(keep=False)]
duplicates

,Rk,Player,Tm,2025-26,2026-27,2027-28,2028-29,2029-30,2030-31,Guaranteed,Player_clean,Total Contract Value,Guaranteed %,Whole Team Name
20,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,NaN
42,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,NaN
64,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,NaN
86,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,NaN
108,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,NaN
130,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,NaN
152,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,NaN
174,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,NaN
196,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,NaN
218,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,NaN


In [13]:
#Step 2: drop duplicate rows
contracts = contracts.drop_duplicates()

#Step 3: validate if any remaining duplicate rows
duplicates = contracts[contracts.duplicated(keep=False)]
duplicates

,Rk,Player,Tm,2025-26,2026-27,2027-28,2028-29,2029-30,2030-31,Guaranteed,Player_clean,Total Contract Value,Guaranteed %,Whole Team Name


In [14]:
#final dataset after transformations
contracts

,Rk,Player,Tm,2025-26,2026-27,2027-28,2028-29,2029-30,2030-31,Guaranteed,Player_clean,Total Contract Value,Guaranteed %,Whole Team Name
0,1,Stephen Curry,GSW,59606817.0,62587158.0,0.0,0.0,0.0,0.0,122193975.0,stephen curry,244387950.0,50.000000,Golden State Warriors
1,2,Joel Embiid,PHI,55224526.0,57985752.0,62624612.0,67263472.0,0.0,0.0,175834890.0,joel embiid,418933252.0,41.972054,Philadelphia 76ers
2,3,Nikola Jokić,DEN,55224526.0,59033114.0,62841702.0,0.0,0.0,0.0,114257640.0,nikola jokić,291356982.0,39.215686,Denver Nuggets
3,4,Kevin Durant,HOU,54708609.0,0.0,0.0,0.0,0.0,0.0,54708609.0,kevin durant,109417218.0,50.000000,Houston Rockets
4,5,Jayson Tatum,BOS,54126450.0,58456566.0,62786682.0,67116798.0,71446914.0,0.0,242486496.0,jayson tatum,556419906.0,43.579767,Boston Celtics
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
400,365,Pelle Larsson,MIA,1955377.0,2296271.0,0.0,0.0,0.0,0.0,0.0,pelle larsson,4251648.0,0.000000,Miami Heat
401,366,E.J. Liddell,PHO,706898.0,706898.0,0.0,0.0,0.0,0.0,1413796.0,ej liddell,2827592.0,50.000000,Phoenix Suns
402,367,Mamadi Diakite,MEM,464050.0,464050.0,0.0,0.0,0.0,0.0,928100.0,mamadi diakite,1856200.0,50.000000,Memphis Grizzlies
403,368,Ricky Rubio,CLE,424672.0,424672.0,0.0,0.0,0.0,0.0,849344.0,ricky rubio,1698688.0,50.000000,Cleveland Cavaliers


# Paragraph of the ethical implications of data wrangling specific to your datasource and the steps you completed 

The transformations to the website data included: standardizing player names and contract salary amounts, created new columns to help with contract calculations, created a column for the proper whole team name, and removed any remaining duplicate rows.  As for legal or regulatory risk, the use of this data does not violate the website Terms of Use.  I do not anticipate any risk from the transformations done to the small amount of columns available from this table.  The primary assumption for this work is that Basketball-Reference.com is a widely-used resource and all available data is accurate and current.  The website Basketball-Reference.com compiles its data from public NBA documents and trusted secondary aggregators.  There are no immediate concerns with ethics regarding the collection of this specific data.  Mitigation of any data concerns can be checked against other similar sources for accuracy.